In [ ]:
import requests
import pandas as pd
import time
import random

# Open Library search API 
# We search by subject/genre and get back book data

response = requests.get(
    "https://openlibrary.org/search.json",
    params={
        "subject": "mystery",
        "limit": 1,
        "fields": "title,author_name,first_publish_year,isbn,publisher,language,subject,number_of_pages_median,cover_i"
    }
)

print("Status:", response.status_code)
data = response.json()
book = data["docs"][0]
print(book)

Status: 200
{'author_name': ['Agatha Christie'], 'cover_i': 11153508, 'first_publish_year': 1924, 'isbn': ['9781684224487', '9798630112330', '9798685423429', '9997509048', '9798830972994', '9781405005968', '9781417616824', '9798589206678', '9788525434050', '1950347346', '142097825X', '9781567230321', '1513221310', '9798739934499', '9798685354655', '9798448851568', '9798445101024', '9798428832709', '9798739944719', '8427285043', '0486837505', '1950330486', '9781950347339', '9789021824307', '9507840095', '9798739933751', '9781556855955', '9798693335080', '9781984899408', '9781572704824', '9780007451555', '0329261517', '9798540304023', '1504763564', '9781636373188', '9781611737738', '9781015230934', '6073914776', '9780312979485', '9780486837505', '9798434348935', '9798479738418', '9798542971926', '2702422217', '9781645940821', '9780062006653', '9798531482600', '0007250622', '9798704765448', '9798445031901', '1572704810', '9780007151660', '9782253024231', '9781504763561', '9798739933492', 

In [3]:
#Check for the avaiable subject

test = requests.get(
    "https://openlibrary.org/search.json",
    params={"subject": "biography", "limit": 1}
)
print(test.status_code)
print(test.json()["numFound"], "books found")

200
988973 books found


In [4]:
df_goodreads = df_goodreads = pd.read_csv("/Users/hinahaq/Downloads/Book_recomendation_Project/goodreads_1000_full.csv")
print(len(df_goodreads))


1000


## Open Library API - Data Collection
Fetching 1000 books from Open Library API using subject-based search. 
Books are checked against the GoodReads dataset to ensure no overlap.

In [ ]:
# NOTE: This cell was already run and data saved to openlibrary_1000.csv
# Run only if you need to regenerate the dataset

existing_titles = set(df_goodreads["title"].str.lower().str.strip())

subjects = [
    "biography", "crime", "history", "psychology", "self_help",
    "travel", "politics", "philosophy", "science", "true_crime",
    "economics", "sociology", "memoir", "journalism", "anthropology"
]

all_books = []

for subject in subjects:
    if len(all_books) >= 1000:
        break

    print(f"Fetching: {subject}...")

    response = requests.get(
        "https://openlibrary.org/search.json",
        params={
            "subject": subject,
            "limit": 100,
            "fields": "title,author_name,first_publish_year,isbn,publisher,language,subject,number_of_pages_median,cover_i,key"
        }
    )

    if response.status_code != 200:
        print(f"  ✗ Failed: {response.status_code}")
        continue

    docs = response.json().get("docs", [])
    print(f"  → {len(docs)} books returned")

    for doc in docs:
        title = doc.get("title", None)
        if not title:
            continue
        if title.lower().strip() in existing_titles:
            continue
        if any(b["title"] == title for b in all_books):
            continue

        book = {
            "title":          title,
            "author":         ", ".join(doc.get("author_name", [])),
            "year_published": doc.get("first_publish_year", None),
            "isbn":           doc.get("isbn", [None])[0],
            "publisher":      doc.get("publisher", [None])[0],
            "language":       ", ".join(doc.get("language", [])),
            "genres":         ", ".join(doc.get("subject", [])[:5]),
            "pages":          doc.get("number_of_pages_median", None),
            "cover_img_url":  f"https://covers.openlibrary.org/b/id/{doc['cover_i']}-L.jpg" if doc.get("cover_i") else None,
            "ol_key":         doc.get("key", None),
            "source":         "Open Library"
        }
        all_books.append(book)

        if len(all_books) >= 1000:
            break

    print(f"  ✓ Total so far: {len(all_books)}")
    time.sleep(1)

df_api = pd.DataFrame(all_books[:1000])
print(f"\nDone! {len(df_api)} books collected.")
print(df_api.head(3))

In [ ]:
#column check
df_api = pd.read_csv("openlibrary_1000.csv")
print(df_api.columns.tolist())
print(df_api.shape)
print(df_api.isnull().sum())

['title', 'author', 'year_published', 'isbn', 'publisher', 'language', 'genres', 'pages', 'cover_img_url', 'ol_key', 'source']
(1000, 11)
title              0
author            16
year_published     3
isbn              33
publisher          5
language           8
genres             0
pages             30
cover_img_url     44
ol_key             0
source             0
dtype: int64


In [18]:
#missing value check
missing_values = df_api.isnull().sum()
print(missing_values)

title              0
author            16
year_published     3
isbn              33
publisher          5
language           8
genres             0
pages             30
cover_img_url     44
ol_key             0
source             0
dtype: int64


In [ ]:
# Look at a few rows where author is missing 
print(df_api[df_api["author"].isna()][["title", "author", "ol_key"]].head(5))

                                                 title author  \
4    Commemorative biographical record of Wayne Cou...    NaN   
31          The Remarkable life of Col. James Gardiner    NaN   
37                The dictionary of national biography    NaN   
192                   Thirty years of the Soviet state    NaN   
241                              P'ing-yang hsien chih    NaN   

                 ol_key  
4      /works/OL271504M  
31   /works/OL15104131M  
37   /works/OL19969194M  
192  /works/OL16597059M  
241  /works/OL17358375M  


In [13]:
print(df_api[df_api["publisher"].isna()][["title", "publisher", "ol_key"]].head(5))

                                title publisher              ol_key
63   Dictionary of National Biography       NaN   /works/OL8305641W
64             The Stranger Beside Me       NaN    /works/OL892102W
76                 The color of water       NaN   /works/OL2956952W
152                 The Quantum Thief       NaN  /works/OL19710455W
890                      Freakonomics       NaN    /works/OL278022W


In [22]:
#check for overlap between the two datasets
scraped_titles = set(df_goodreads["title"].str.lower().str.strip())
api_titles = set(df_api["title"].str.lower().str.strip())

overlap = scraped_titles.intersection(api_titles)
print(f"Overlapping books: {len(overlap)}")
print(overlap)

Overlapping books: 0
set()


In [20]:
#redefining functions 
def fetch_book_details(ol_key):
    url = f"https://openlibrary.org{ol_key}.json"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code != 200:
            return {}
        data = response.json()
        
        desc = data.get("description", None)
        if isinstance(desc, dict):
            desc = desc.get("value", None)
            
        return {"description": desc}
    except:
        return {}


def fetch_author_details(ol_key):
    url = f"https://openlibrary.org{ol_key}.json"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code != 200:
            return {}
        data = response.json()
        
        authors = data.get("authors", [])
        if not authors:
            return {}
            
        author_key = authors[0].get("author", {}).get("key", None)
        if not author_key:
            return {}
            
        author_response = requests.get(f"https://openlibrary.org{author_key}.json", timeout=10)
        if author_response.status_code == 200:
            return {"author": author_response.json().get("name", None)}
        return {}
    except:
        return {}

In [ ]:
#test for getting missing information
test_key = "/works/OL271504M"
print("Testing author fetch:")
print(fetch_author_details(test_key))

print("\nTesting description fetch:")
print(fetch_book_details(test_key))

Testing author fetch:
{}

Testing description fetch:
{'description': None}


In [23]:
# Test on a book we know Open Library has full details for
test_key = "/works/OL892102W"  # The Stranger Beside Me (Ted Bundy book)

print("Testing author fetch:")
print(fetch_author_details(test_key))

print("\nTesting description fetch:")
print(fetch_book_details(test_key))

Testing author fetch:
{'author': 'Ann Rule'}

Testing description fetch:
{'description': "There are actually two stories here: one describes the gradual disintegration of a seemingly normal, affable, brilliant man into a sexual psychopath so evil, so methodical in his vicious killings, that one wonders if he was at all human. The other story is that of Ann Rule herself, a decent, hard-working, middle-aged mother of four who meets and befriends a nice young man working beside her in a crisis clinic. A man she regards as a younger brother; a man she views as a close and trusted friend. The slow but inexorable realization on Rule's part that this man is in fact an unspeakably violent serial killer is as painful to read as it was for her to experience.\r\n\r\nEach victim is described in terms of such respect and such anguish that even a family member, I think, can feel that his or her daughter has been given a chance to shine, a chance to be more than a victim, more than a nameless number 

In [24]:
#check how many books have complete discription and author info

# Test on 5 books to see what percentage have descriptions
test_keys = df_api["ol_key"].head(10).tolist()

complete = 0
for key in test_keys:
    result = fetch_book_details(key)
    has_desc = result.get("description") is not None
    print(f"{key} → description: {has_desc}")
    if has_desc:
        complete += 1
    time.sleep(1)

print(f"\n{complete}/10 have descriptions")


/works/OL1037811W → description: True
/works/OL27294413W → description: True
/works/OL528094W → description: True
/works/OL26663382W → description: True
/works/OL271504M → description: False
/works/OL158041W → description: True
/works/OL1847023W → description: True
/works/OL20715162W → description: True
/works/OL69178W → description: True
/works/OL2625422W → description: True

9/10 have descriptions


In [25]:
df_api = pd.read_csv("openlibrary_1000.csv")
existing_titles = set(df_api["title"].str.lower().str.strip())

# Pool of replacement subjects to draw from
replacement_subjects = ["literature", "classics", "fiction", "drama", "poetry",
                        "business", "health", "education", "law", "religion"]

def get_replacement_book(existing_titles):
    """Fetch one complete replacement book from Open Library"""
    for subject in replacement_subjects:
        response = requests.get(
            "https://openlibrary.org/search.json",
            params={
                "subject": subject,
                "limit": 50,
                "fields": "title,author_name,first_publish_year,isbn,publisher,language,subject,number_of_pages_median,cover_i,key"
            }
        )
        if response.status_code != 200:
            continue

        for doc in response.json().get("docs", []):
            title = doc.get("title", None)
            if not title or title.lower().strip() in existing_titles:
                continue

            ol_key = doc.get("key", None)
            if not ol_key:
                continue

            # Check it has a description
            details = fetch_book_details(ol_key)
            if not details.get("description"):
                continue

            # Build the book dict
            book = {
                "title":          title,
                "author":         ", ".join(doc.get("author_name", [])),
                "year_published": doc.get("first_publish_year", None),
                "isbn":           doc.get("isbn", [None])[0],
                "publisher":      doc.get("publisher", [None])[0],
                "language":       ", ".join(doc.get("language", [])),
                "genres":         ", ".join(doc.get("subject", [])[:5]),
                "pages":          doc.get("number_of_pages_median", None),
                "cover_img_url":  f"https://covers.openlibrary.org/b/id/{doc['cover_i']}-L.jpg" if doc.get("cover_i") else None,
                "ol_key":         ol_key,
                "source":         "Open Library",
                "description":    details["description"]
            }
            existing_titles.add(title.lower().strip())
            return book

    return None


# Main enrichment loop with replacement
rows_to_drop = []

for i, row in df_api.iterrows():
    print(f"[{i+1}/1000] {row['title']}")

    ol_key = row["ol_key"]
    if not ol_key:
        rows_to_drop.append(i)
        continue

    # Fetch description
    details = fetch_book_details(ol_key)
    description = details.get("description", None)

    # Fix missing author
    author = row["author"]
    if pd.isna(author) or author == "":
        author_details = fetch_author_details(ol_key)
        author = author_details.get("author", None)
        df_api.at[i, "author"] = author

    if not description:
        print(f"  ✗ No description — replacing...")
        replacement = get_replacement_book(existing_titles)
        if replacement:
            for col, val in replacement.items():
                df_api.at[i, col] = val
            print(f"  ✓ Replaced with: {replacement['title']}")
        else:
            rows_to_drop.append(i)
    else:
        df_api.at[i, "description"] = description

    # Save every 50 books
    if (i + 1) % 50 == 0:
        df_api.to_csv("openlibrary_1000.csv", index=False)
        print(f"  ✓ Progress saved at {i+1} books")

    time.sleep(1)

# Final save
df_api.to_csv("openlibrary_1000.csv", index=False)
print(f"\nDone! {len(df_api)} books saved.")
print(f"Rows flagged for drop: {len(rows_to_drop)}")

[1/1000] Wings of Fire
[2/1000] The Diary of a Young Girl- Anne Frank
[3/1000] Autobiography of a Yogi
[4/1000] I'm Glad My Mom Died
[5/1000] Commemorative biographical record of Wayne County, Ohio, containing biographical sketches of prominent and representative citizens, and of many of the early settled families 
  ✗ No description — replacing...
  ✓ Replaced with: Reminders of Him
[6/1000] Fear and Loathing in Las Vegas
[7/1000] The worldly philosophers
[8/1000] Persepolis 1 & 2
[9/1000] Narrative of the life of Frederick Douglass
[10/1000] 走ることについて語るときに僕の語ること
[11/1000] Just Kids
[12/1000] Born a Crime
[13/1000] Pimp
[14/1000] Gifted hands
[15/1000] Ikigai
  ✗ No description — replacing...
  ✓ Replaced with: The Prophet
[16/1000] When Breath Becomes Air
[17/1000] All Creatures Great and Small (All Creatures Great and Small #1-2)
[18/1000] Elon Musk
[19/1000] On Writing
[20/1000] Livro do Desassossego
[21/1000] Elvis and me
[22/1000] Kitchen Confidential
[23/1000] I Know Why the Cage

In [ ]:
#cheecking the languge breakdown
print("=== GOODREADS columns ===")
print(df_goodreads.columns.tolist())

print("\n=== OPEN LIBRARY language breakdown ===")
df_api = pd.read_csv("openlibrary_1000.csv")
print(df_api["language"].value_counts())

=== GOODREADS columns ===
['rank', 'title', 'author', 'avg_rating', 'num_ratings', 'cover_img_url', 'goodreads_url', 'source', 'year_published', 'genres', 'description', 'pages', 'series']

=== OPEN LIBRARY language breakdown ===
language
eng                                                 325
eng, spa                                             25
spa, eng                                             19
ind                                                  19
eng, ger                                             17
                                                   ... 
por, eng                                              1
gem, tel, eng, tha, spa, rus                          1
eng, kor, jpn, vie, spa, chi, por, rus, fre, pan      1
eng, vie, dut                                         1
ger, eng, por                                         1
Name: count, Length: 512, dtype: int64


In [28]:
# Books with no English at all
non_english = df_api[~df_api["language"].str.contains("eng", na=False)]
print(f"Books with no English: {len(non_english)}")
print(non_english["language"].value_counts())

Books with no English: 29
language
ind              19
hin               1
spa, ger, por     1
tgl               1
may               1
dut               1
nor, swe          1
Name: count, dtype: int64


In [31]:
#function defination
# Test the new strict replacement function
existing_titles = set(df_api["title"].str.lower().str.strip())
test_replacement = get_replacement_book(existing_titles)
print(test_replacement)

{'title': 'Curriculum development', 'author': 'Jon Wiles, Jon W. Wiles, Joseph C. Bondi, Joseph Bondi', 'year_published': 1979, 'isbn': '9780131380875', 'publisher': 'Merrill/Prentice Hall', 'language': 'eng', 'genres': 'Curricula, Curriculum planning, Education, Curriculum planning & development, Education (General)', 'pages': 384, 'cover_img_url': 'https://covers.openlibrary.org/b/id/4957815-L.jpg', 'ol_key': '/works/OL1941352W', 'source': 'Open Library', 'description': 'A "standard" in curriculum books, Wiles/BondiCurriculum Developmentcontinues the historic strength of the book–history, philosophy, and foundations of curriculum development–and addresses new trends in curriculum development resulting from standards and emerging technologies. This respected author team examines how technology and standards-based education are impacting the future directions of education and discusses how to preserve historic school values while responding to current educational trends. This edition f

In [32]:
#replacing the non-English books with English ones 
non_english_indices = df_api[~df_api["language"].str.contains("eng", na=False)].index.tolist()
print(f"Replacing {len(non_english_indices)} non-English books...")

existing_titles = set(df_api["title"].str.lower().str.strip())

for i in non_english_indices:
    old_title = df_api.at[i, "title"]
    print(f"  Replacing: {old_title}")
    
    replacement = get_replacement_book(existing_titles)
    if replacement:
        for col, val in replacement.items():
            df_api.at[i, col] = val
        print(f"  ✓ Replaced with: {replacement['title']}")
    else:
        print(f"  ✗ No replacement found for index {i}")

    time.sleep(1)

df_api.to_csv("openlibrary_1000.csv", index=False)
print("\nDone!")

# Verify
non_english_after = df_api[~df_api["language"].str.contains("eng", na=False)]
print(f"Non-English books remaining: {len(non_english_after)}")

Replacing 29 non-English books...
  Replacing: Gunaho ka Devta
  ✓ Replaced with: Curriculum development
  Replacing: Seporsi mie ayam sebelum mati
  ✓ Replaced with: Up from Slavery
  Replacing: Vidas secas
  ✓ Replaced with: Dumbing Us Down
  Replacing: The Big Book of Serial Killers
  ✓ Replaced with: Measurement and Assessment in Teaching
  Replacing: Mga gunita ng himagsikan
  ✓ Replaced with: Pedagogy of the Oppressed
  Replacing: Psikologi remaja
  ✓ Replaced with: Multicultural Education
  Replacing: Matahari
  ✓ Replaced with: Emile or Education
  Replacing: Kualitas pelayanan publik
  ✓ Replaced with: Basic principles of curriculum and instruction
  Replacing: Pancasila
  ✓ Replaced with: Childcraft
  Replacing: Catatan seorang demonstran
  ✓ Replaced with: The Abolition of Man
  Replacing: Implementasi kebijakan publik
  ✓ Replaced with: Education For Critical Consciousness (Continuum Impacts)
  Replacing: Aku bangga jadi anak PKI
  ✓ Replaced with: Ancient law, its connecti

In [33]:
# Check overlap between GoodReads and Open Library datasets
df_goodreads = pd.read_csv("goodreads_1000_full.csv")
df_api = pd.read_csv("openlibrary_1000.csv")

goodreads_titles = set(df_goodreads["title"].str.lower().str.strip())
api_titles = set(df_api["title"].str.lower().str.strip())

overlap = goodreads_titles.intersection(api_titles)
print(f"Overlapping books between datasets: {len(overlap)}")
if overlap:
    print(overlap)

Overlapping books between datasets: 45
{'leaves of grass', 'the silent patient', 'misery', 'the odyssey', 'death of a salesman', 'the silmarillion', 'meditations', 'the outsiders', 'tao te ching', 'mythology', 'the sun also rises', 'emma', 'the scarlet letter', 'inferno', 'the glass menagerie', 'the crucible', 'king lear', 'macbeth', 'the night before christmas', 'siddhartha', 'the picture of dorian gray', 'a streetcar named desire', 'hamlet', 'the prophet', 'a light in the attic', 'the awakening', 'verity', 'local woman missing', 'the great gatsby', 'the secret garden', 'mrs. dalloway', 'the war of the worlds', 'paradise lost', 'white fang', 'where the red fern grows', 'the time machine', 'girl, interrupted', 'wuthering heights', 'romeo and juliet', 'on the road', 'animal farm', 'the importance of being earnest', 'julius caesar', 'the merchant of venice', 'a little princess'}


In [34]:
# Remove overlapping books from Open Library dataset
overlap_indices = df_api[df_api["title"].str.lower().str.strip().isin(overlap)].index.tolist()
print(f"Removing {len(overlap_indices)} overlapping books from Open Library dataset")

df_api = df_api.drop(overlap_indices).reset_index(drop=True)
print(f"Open Library dataset after removal: {len(df_api)} books")

# Now fetch 45 replacements
existing_titles = set(df_goodreads["title"].str.lower().str.strip()) | set(df_api["title"].str.lower().str.strip())

replacements = []
extra_subjects = ["cooking", "gardening", "sports", "music", "architecture",
                  "mathematics", "medicine", "astronomy", "geology", "linguistics"]

for subject in extra_subjects:
    if len(replacements) >= 45:
        break
        
    print(f"Fetching: {subject}...")
    response = requests.get(
        "https://openlibrary.org/search.json",
        params={
            "subject": subject,
            "limit": 100,
            "fields": "title,author_name,first_publish_year,isbn,publisher,language,subject,number_of_pages_median,cover_i,key"
        }
    )
    
    if response.status_code != 200:
        continue
        
    for doc in response.json().get("docs", []):
        if len(replacements) >= 45:
            break
            
        title = doc.get("title", None)
        if not title or title.lower().strip() in existing_titles:
            continue
            
        languages = doc.get("language", [])
        if "eng" not in languages:
            continue
            
        author    = ", ".join(doc.get("author_name", []))
        year      = doc.get("first_publish_year", None)
        isbn      = doc.get("isbn", [None])[0]
        publisher = doc.get("publisher", [None])[0]
        pages     = doc.get("number_of_pages_median", None)
        cover_i   = doc.get("cover_i", None)
        ol_key    = doc.get("key", None)
        
        if not all([author, year, isbn, publisher, pages, cover_i, ol_key]):
            continue
        
        details = fetch_book_details(ol_key)
        if not details.get("description"):
            continue
            
        book = {
            "title":          title,
            "author":         author,
            "year_published": year,
            "isbn":           isbn,
            "publisher":      publisher,
            "language":       ", ".join(languages),
            "genres":         ", ".join(doc.get("subject", [])[:5]),
            "pages":          pages,
            "cover_img_url":  f"https://covers.openlibrary.org/b/id/{cover_i}-L.jpg",
            "ol_key":         ol_key,
            "source":         "Open Library",
            "description":    details["description"]
        }
        replacements.append(book)
        existing_titles.add(title.lower().strip())
        print(f"  ✓ Added: {title}")
        
    time.sleep(1)

print(f"\nFound {len(replacements)} replacements")
df_replacements = pd.DataFrame(replacements)
df_api = pd.concat([df_api, df_replacements], ignore_index=True)
print(f"Open Library dataset final count: {len(df_api)}")

df_api.to_csv("openlibrary_1000.csv", index=False)
print("Saved!")

Removing 45 overlapping books from Open Library dataset
Open Library dataset after removal: 955 books
Fetching: cooking...
  ✓ Added: How to Cook Everything
  ✓ Added: Boston Cooking-School cook book
  ✓ Added: Joy of Cooking
  ✓ Added: Salt, Fat, Acid, Heat
  ✓ Added: White Heat
  ✓ Added: Odd Thomas
  ✓ Added: Le répertoire de la cuisine
  ✓ Added: Guide culinaire
  ✓ Added: A Book of Mediterranean Food
  ✓ Added: Cooking Basics for Dummies
  ✓ Added: The Professional Chef
  ✓ Added: Betty Crocker Cookbook
  ✓ Added: How to Be a Domestic Goddess
  ✓ Added: Odd Hours
  ✓ Added: French Country Cooking (Cookery Library)
  ✓ Added: Mastering the art of French cooking
  ✓ Added: Panaderia y Reposteria para profesionales/Professional Baking
  ✓ Added: The I hate to cook book
  ✓ Added: French Chef Cookbook, The
  ✓ Added: New York Times Cook Book
  ✓ Added: French provincial cooking
  ✓ Added: The flavor bible
  ✓ Added: Original White House Cookbook
  ✓ Added: Ma cuisine
  ✓ Added: Frugal

In [35]:
# Find and remove the overlap from Open Library dataset
print(overlap)

overlap_index = df_api[df_api["title"].str.lower().str.strip().isin(overlap)].index.tolist()
df_api = df_api.drop(overlap_index).reset_index(drop=True)
print(f"Open Library after removal: {len(df_api)}")

{'leaves of grass', 'the silent patient', 'misery', 'the odyssey', 'death of a salesman', 'the silmarillion', 'meditations', 'the outsiders', 'tao te ching', 'mythology', 'the sun also rises', 'emma', 'the scarlet letter', 'inferno', 'the glass menagerie', 'the crucible', 'king lear', 'macbeth', 'the night before christmas', 'siddhartha', 'the picture of dorian gray', 'a streetcar named desire', 'hamlet', 'the prophet', 'a light in the attic', 'the awakening', 'verity', 'local woman missing', 'the great gatsby', 'the secret garden', 'mrs. dalloway', 'the war of the worlds', 'paradise lost', 'white fang', 'where the red fern grows', 'the time machine', 'girl, interrupted', 'wuthering heights', 'romeo and juliet', 'on the road', 'animal farm', 'the importance of being earnest', 'julius caesar', 'the merchant of venice', 'a little princess'}
Open Library after removal: 1000


In [36]:
print(f"Open Library: {len(df_api)}")

# Verify no overlap now
goodreads_titles = set(df_goodreads["title"].str.lower().str.strip())
api_titles = set(df_api["title"].str.lower().str.strip())
overlap_check = goodreads_titles.intersection(api_titles)
print(f"Overlapping books remaining: {len(overlap_check)}")

Open Library: 1000
Overlapping books remaining: 0


In [ ]:
API_KEY = #hiding the API key for security reasons, but it is needed to run the code below

def get_google_books_data(title, author):
    """Search Google Books and return rating, url, isbn"""
    try:
        query = f"{title} {author}"
        response = requests.get(
            "https://www.googleapis.com/books/v1/volumes",
            params={"q": query, "key": API_KEY, "maxResults": 1}
        )
        if response.status_code != 200:
            return {}
        
        items = response.json().get("items", [])
        if not items:
            return {}
        
        book = items[0]
        info = book.get("volumeInfo", {})
        
        # ISBN
        isbn = None
        for identifier in info.get("industryIdentifiers", []):
            if identifier["type"] == "ISBN_13":
                isbn = identifier["identifier"]
                break
        
        return {
            "avg_rating":    info.get("averageRating", None),
            "num_ratings":   info.get("ratingsCount", None),
            "google_url":    info.get("infoLink", None),
            "isbn":          isbn
        }
    except:
        return {}

# Test on 1 book
result = get_google_books_data("Wings of Fire", "A.P.J. Abdul Kalam")
print(result)

{'avg_rating': 4.5, 'num_ratings': 182, 'google_url': 'http://books.google.de/books?id=c3qmIZtWUjAC&dq=Wings+of+Fire+A.P.J.+Abdul+Kalam&hl=&source=gbs_api', 'isbn': '9788173711466'}


In [38]:
#fetching the missing variables avg_rating, url, etc
df_api = pd.read_csv("openlibrary_1000.csv")

# Add empty columns
df_api["avg_rating"] = None
df_api["num_ratings"] = None
df_api["google_url"] = None

for i, row in df_api.iterrows():
    print(f"[{i+1}/1000] {row['title']}")
    
    result = get_google_books_data(row["title"], row["author"])
    
    df_api.at[i, "avg_rating"]  = result.get("avg_rating", None)
    df_api.at[i, "num_ratings"] = result.get("num_ratings", None)
    df_api.at[i, "google_url"]  = result.get("google_url", None)
    
    # Only fill isbn if missing
    if pd.isna(row["isbn"]) and result.get("isbn"):
        df_api.at[i, "isbn"] = result.get("isbn")

    # Save every 50 books
    if (i + 1) % 50 == 0:
        df_api.to_csv("openlibrary_1000.csv", index=False)
        print(f"  ✓ Progress saved at {i+1} books")

    time.sleep(1)

df_api.to_csv("openlibrary_1000.csv", index=False)
print("Done!")

[1/1000] Wings of Fire
[2/1000] The Diary of a Young Girl- Anne Frank
[3/1000] Autobiography of a Yogi
[4/1000] I'm Glad My Mom Died
[5/1000] Reminders of Him
[6/1000] Fear and Loathing in Las Vegas
[7/1000] The worldly philosophers
[8/1000] Persepolis 1 & 2
[9/1000] Narrative of the life of Frederick Douglass
[10/1000] 走ることについて語るときに僕の語ること
[11/1000] Just Kids
[12/1000] Born a Crime
[13/1000] Pimp
[14/1000] Gifted hands
[15/1000] When Breath Becomes Air
[16/1000] All Creatures Great and Small (All Creatures Great and Small #1-2)
[17/1000] Elon Musk
[18/1000] On Writing
[19/1000] Livro do Desassossego
[20/1000] Elvis and me
[21/1000] Kitchen Confidential
[22/1000] I Know Why the Caged Bird Sings
[23/1000] The Man Who Loved Only Numbers
[24/1000] Leonardo da Vinci
[25/1000] The Woman in Me
[26/1000] Can't Hurt Me
[27/1000] Einstein
[28/1000] I am Malala
[29/1000] Twelve against the gods
[30/1000] The Story of Philosophy
[31/1000] The Norton Anthology of English Literature
[32/1000] Maybe 

In [39]:
# Check for non-ASCII characters in titles (catches Japanese, Arabic, Cyrillic, Greek etc.)
non_english_titles = df_api[df_api["title"].str.contains(
    r'[^\x00-\x7F]', na=False, regex=True
)]
print(f"Non-English titles: {len(non_english_titles)}")
print(non_english_titles["title"].tolist())

Non-English titles: 47
['走ることについて語るときに僕の語ること', 'कामसूत्र', 'Once You’re Mine', 'Das Schloß', 'Mördare utan ansikte', 'Преступление и наказание', 'Le città invisibili', 'Ὀδύσσεια', 'La conquête du pain', 'The Eagle of the Ninth. 1400 Grundwörter.', 'Братья Карамазовы', 'Siete breves lecciones de física', 'Анна Каренина', 'Memórias póstumas de Brás Cubas', 'Götter, Gräber und Gelehrte', '嫌われる勇気', 'Записки изъ подполья', 'Tuḥfat al-nuẓẓār fī gharāʾib al-amṣār wa-ʻajāʾib al-asfār', 'Οἰδίπους Τύραννος (Oidípous Týrannos)', 'Πολιτικά (Politiká)', 'Мастер и Маргарита', 'Архипелаг ГУЛАГ', 'πολιτεία', 'Conversación en La Catedral', 'Salomé', 'Culpa mía', 'Ἀπολογία Σωκράτους', 'Discours de la méthode', 'Φαίδων', 'Дядя Ваня', 'Veinte poemas de amor y una canción desesperada', 'Συμπόσιον', 'Fröhliche Wissenschaft', 'Εὐθύφρων / Κρίτων / Μένων / Φαίδων / Ἀπολογία Σωκράτους', 'Μένων', 'Différence et répétition', 'Die Geburt der Tragödie', 'Понедельник начинается в субботу; Пикник на обочине', 'Megha

In [42]:
df_api = pd.read_csv("openlibrary_1000.csv")
existing_titles = set(df_goodreads["title"].str.lower().str.strip()) | set(df_api["title"].str.lower().str.strip())

non_english_idx = df_api[df_api["title"].str.contains(r'[^\x00-\x7F]', na=False, regex=True)].index.tolist()
print(f"Replacing {len(non_english_idx)} books...")

for i in non_english_idx:
    old_title = df_api.at[i, "title"]
    print(f"  Replacing: {old_title}")
    
    try:
        replacement = get_replacement_book(existing_titles)
        if replacement:
            for col, val in replacement.items():
                df_api.at[i, col] = val
            print(f"  ✓ Replaced with: {replacement['title']}")
        else:
            print(f"  ✗ No replacement found")
    except Exception as e:
        print(f"  ✗ Error: {e} — waiting 60 seconds...")
        time.sleep(60)  # wait longer if connection drops
        continue

    # Save after EVERY book so we don't lose progress
    df_api.to_csv("openlibrary_1000.csv", index=False)
    time.sleep(5)  # longer delay between books

print("\nDone!")
remaining = df_api[df_api["title"].str.contains(r'[^\x00-\x7F]', na=False, regex=True)]
print(f"Non-English titles remaining: {len(remaining)}")

Replacing 47 books...
  Replacing: 走ることについて語るときに僕の語ること
  ✓ Replaced with: Zukunft einer Illusion
  Replacing: कामसूत्र
  ✓ Replaced with: The Teachings of Don Juan
  Replacing: Once You’re Mine
  ✓ Replaced with: God Is Not One
  Replacing: Das Schloß
  ✓ Replaced with: People of the lie
  Replacing: Mördare utan ansikte
  ✓ Replaced with: Are You There God? It's Me, Margaret.
  Replacing: Преступление и наказание
  ✗ Error: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')) — waiting 60 seconds...
  Replacing: Le città invisibili
  ✓ Replaced with: The Book of the Dead
  Replacing: Ὀδύσσεια
  ✓ Replaced with: The World's Religions
  Replacing: La conquête du pain
  ✓ Replaced with: Saint Francis of Assisi
  Replacing: The Eagle of the Ninth. 1400 Grundwörter.
  ✓ Replaced with: Beyond theology
  Replacing: Братья Карамазовы
  ✓ Replaced with: Histoire des croyances et des idées religieuses
  Replacing: Siete breves lecciones de física
  ✓ Replaced with: Mira

In [43]:
remaining = df_api[df_api["title"].str.contains(r'[^\x00-\x7F]', na=False, regex=True)]
print(remaining["title"].tolist())
print(remaining.index.tolist())

['Преступление и наказание', 'Histoire des croyances et des idées religieuses', 'Mas̲navī', 'Μένων']
[173, 232, 466, 654]


In [44]:
#Replacing the remaning 4 non-English books with English ones using the same function as before
existing_titles = set(df_goodreads["title"].str.lower().str.strip()) | set(df_api["title"].str.lower().str.strip())

for i in [173, 232, 466, 654]:
    old_title = df_api.at[i, "title"]
    print(f"  Replacing: {old_title}")
    
    try:
        replacement = get_replacement_book(existing_titles)
        if replacement:
            for col, val in replacement.items():
                df_api.at[i, col] = val
            print(f"  ✓ Replaced with: {replacement['title']}")
            df_api.to_csv("openlibrary_1000.csv", index=False)
        else:
            print(f"  ✗ No replacement found")
    except Exception as e:
        print(f"  ✗ Error: {e} — waiting 60 seconds...")
        time.sleep(60)
    
    time.sleep(5)

# Final check
remaining = df_api[df_api["title"].str.contains(r'[^\x00-\x7F]', na=False, regex=True)]
print(f"\nNon-English titles remaining: {len(remaining)}")

  Replacing: Преступление и наказание
  ✓ Replaced with: कामसूत्र
  Replacing: Histoire des croyances et des idées religieuses
  ✓ Replaced with: Once You’re Mine
  Replacing: Mas̲navī
  ✓ Replaced with: Das Schloß
  Replacing: Μένων
  ✓ Replaced with: Le città invisibili

Non-English titles remaining: 4


replcemnat books were non english, replacing them again.

In [45]:
def get_replacement_book(existing_titles):
    """Fetch one complete replacement book - must have all key fields and English title"""
    for subject in replacement_subjects:
        response = requests.get(
            "https://openlibrary.org/search.json",
            params={
                "subject": subject,
                "limit": 50,
                "fields": "title,author_name,first_publish_year,isbn,publisher,language,subject,number_of_pages_median,cover_i,key"
            }
        )
        if response.status_code != 200:
            continue

        for doc in response.json().get("docs", []):
            title = doc.get("title", None)
            if not title:
                continue
            
            # Skip non-ASCII titles
            if not title.isascii():
                continue
                
            if title.lower().strip() in existing_titles:
                continue

            languages = doc.get("language", [])
            if "eng" not in languages:
                continue

            author    = ", ".join(doc.get("author_name", []))
            year      = doc.get("first_publish_year", None)
            isbn      = doc.get("isbn", [None])[0]
            publisher = doc.get("publisher", [None])[0]
            pages     = doc.get("number_of_pages_median", None)
            cover_i   = doc.get("cover_i", None)
            ol_key    = doc.get("key", None)

            if not all([author, year, isbn, publisher, pages, cover_i, ol_key]):
                continue

            details = fetch_book_details(ol_key)
            if not details.get("description"):
                continue

            book = {
                "title":          title,
                "author":         author,
                "year_published": year,
                "isbn":           isbn,
                "publisher":      publisher,
                "language":       ", ".join(languages),
                "genres":         ", ".join(doc.get("subject", [])[:5]),
                "pages":          pages,
                "cover_img_url":  f"https://covers.openlibrary.org/b/id/{cover_i}-L.jpg",
                "ol_key":         ol_key,
                "source":         "Open Library",
                "description":    details["description"]
            }
            existing_titles.add(title.lower().strip())
            return book

    return None


# Now replace the 4 remaining
existing_titles = set(df_goodreads["title"].str.lower().str.strip()) | set(df_api["title"].str.lower().str.strip())

remaining_idx = df_api[df_api["title"].str.contains(r'[^\x00-\x7F]', na=False, regex=True)].index.tolist()
print(f"Replacing {len(remaining_idx)} books...")

for i in remaining_idx:
    old_title = df_api.at[i, "title"]
    print(f"  Replacing: {old_title}")
    
    try:
        replacement = get_replacement_book(existing_titles)
        if replacement:
            for col, val in replacement.items():
                df_api.at[i, col] = val
            print(f"  ✓ Replaced with: {replacement['title']}")
            df_api.to_csv("openlibrary_1000.csv", index=False)
        else:
            print(f"  ✗ No replacement found")
    except Exception as e:
        print(f"  ✗ Error: {e} — waiting 60 seconds...")
        time.sleep(60)
    
    time.sleep(5)

remaining = df_api[df_api["title"].str.contains(r'[^\x00-\x7F]', na=False, regex=True)]
print(f"\nNon-English titles remaining: {len(remaining)}")

Replacing 4 books...
  Replacing: कामसूत्र
  ✓ Replaced with: The Garden of Evening Mists
  Replacing: Once You’re Mine
  ✓ Replaced with: The No-Garden Gardener
  Replacing: Das Schloß
  ✓ Replaced with: Vegetable gardening for dummies
  Replacing: Le città invisibili
  ✓ Replaced with: Grumpy Darling

Non-English titles remaining: 0


In [46]:
# checking for, duplicats, shape, colums, overlap and missing values

df_goodreads = pd.read_csv("goodreads_1000_full.csv")
df_api = pd.read_csv("openlibrary_1000.csv")

print("=== SHAPES ===")
print(f"GoodReads: {df_goodreads.shape}")
print(f"Open Library: {df_api.shape}")

print("\n=== COLUMNS ===")
print("GoodReads:", df_goodreads.columns.tolist())
print("Open Library:", df_api.columns.tolist())

print("\n=== DUPLICATES WITHIN EACH DATASET ===")
print(f"GoodReads duplicates: {df_goodreads['title'].duplicated().sum()}")
print(f"Open Library duplicates: {df_api['title'].duplicated().sum()}")

print("\n=== OVERLAP BETWEEN DATASETS ===")
gr_titles = set(df_goodreads["title"].str.lower().str.strip())
api_titles = set(df_api["title"].str.lower().str.strip())
overlap = gr_titles.intersection(api_titles)
print(f"Overlapping books: {len(overlap)}")

print("\n=== NON-ENGLISH TITLES ===")
gr_non_english = df_goodreads[df_goodreads["title"].str.contains(r'[^\x00-\x7F]', na=False, regex=True)]
api_non_english = df_api[df_api["title"].str.contains(r'[^\x00-\x7F]', na=False, regex=True)]
print(f"GoodReads non-English titles: {len(gr_non_english)}")
print(f"Open Library non-English titles: {len(api_non_english)}")

print("\n=== MISSING VALUES - GOODREADS ===")
print(df_goodreads.isnull().sum())

print("\n=== MISSING VALUES - OPEN LIBRARY ===")
print(df_api.isnull().sum())

=== SHAPES ===
GoodReads: (1004, 13)
Open Library: (1000, 15)

=== COLUMNS ===
GoodReads: ['rank', 'title', 'author', 'avg_rating', 'num_ratings', 'cover_img_url', 'goodreads_url', 'source', 'year_published', 'genres', 'description', 'pages', 'series']
Open Library: ['title', 'author', 'year_published', 'isbn', 'publisher', 'language', 'genres', 'pages', 'cover_img_url', 'ol_key', 'source', 'description', 'avg_rating', 'num_ratings', 'google_url']

=== DUPLICATES WITHIN EACH DATASET ===
GoodReads duplicates: 0
Open Library duplicates: 0

=== OVERLAP BETWEEN DATASETS ===
Overlapping books: 1

=== NON-ENGLISH TITLES ===
GoodReads non-English titles: 46
Open Library non-English titles: 0

=== MISSING VALUES - GOODREADS ===
rank                0
title               0
author              0
avg_rating         22
num_ratings         0
cover_img_url       0
goodreads_url       0
source              0
year_published      7
genres              7
description         0
pages               7
series

In [47]:
gr_non_english = df_goodreads[df_goodreads["title"].str.contains(r'[^\x00-\x7F]', na=False, regex=True)]
print(gr_non_english["title"].tolist())

['Alice’s Adventures in Wonderland / Through the Looking-Glass', 'Les Misérables', 'Charlotte’s Web', "Ender’s Game (Ender's Saga, #1)", "The Handmaid’s Tale (The Handmaid's Tale, #1)", 'One Flew Over the Cuckoo’s Nest', 'The Titan’s Curse (Percy Jackson and the Olympians, #3)', "The Ultimate Hitchhiker’s Guide to the Galaxy (Hitchhiker's Guide to the Galaxy, #1-5)", 'A Midsummer Night’s Dream', 'Tess of the D’Urbervilles', 'Oh, the Places You’ll Go!', 'Schindler’s List', 'Shōgun (Asian Saga, #1)', 'Cat’s Cradle', 'Marley and Me: Life and Love With the World’s Worst Dog', 'She’s Come Undone', 'The Girl Who Kicked the Hornet’s Nest (Millennium, #3)', 'Uncle Tom’s Cabin', 'Blood River: A Journey to Africa’s Broken Heart', 'The Once and Future King (The Once and Future King, #1–5)', 'Lamb: The Gospel According to Biff, Christ’s Childhood Pal', 'Are You There God? It’s Me, Margaret', 'Sophie’s World', 'Sophie’s Choice', 'The Magician’s Nephew (Chronicles of Narnia, #6)', 'Bridget Jones’s D

In [48]:
print(df_api.isnull().sum())

title               0
author              1
year_published      0
isbn                4
publisher           5
language            0
genres              0
pages              12
cover_img_url       4
ol_key              0
source              0
description         0
avg_rating        732
num_ratings       732
google_url         12
dtype: int64


In [ ]:
# Check for overlap again before final save
gr_titles = set(df_goodreads["title"].str.lower().str.strip())
api_titles = set(df_api["title"].str.lower().str.strip())
overlap = gr_titles.intersection(api_titles)
print("Overlapping book:", overlap)

# Remove from Open Library
overlap_idx = df_api[df_api["title"].str.lower().str.strip().isin(overlap)].index.tolist()
df_api = df_api.drop(overlap_idx).reset_index(drop=True)
print(f"Open Library after removal: {len(df_api)}")

# Fetch 1 replacement
existing_titles = set(df_goodreads["title"].str.lower().str.strip()) | set(df_api["title"].str.lower().str.strip())
replacement = get_replacement_book(existing_titles)
if replacement:
    df_api = pd.concat([df_api, pd.DataFrame([replacement])], ignore_index=True)
    print(f"Replaced with: {replacement['title']}")

df_api.to_csv("openlibrary_1000.csv", index=False)
print(f"Final Open Library count: {len(df_api)}")

Overlapping book: {'tender is the night'}
Open Library after removal: 999
Replaced with: The Deal / The Mistake / The Score / The Goal / The Legacy
Final Open Library count: 1000


In [50]:
# Final verification
gr_titles = set(df_goodreads["title"].str.lower().str.strip())
api_titles = set(df_api["title"].str.lower().str.strip())

overlap = gr_titles.intersection(api_titles)
print(f"Overlap between datasets: {len(overlap)}")
print(f"GoodReads duplicates: {df_goodreads['title'].duplicated().sum()}")
print(f"Open Library duplicates: {df_api['title'].duplicated().sum()}")
print(f"GoodReads shape: {df_goodreads.shape}")
print(f"Open Library shape: {df_api.shape}")

Overlap between datasets: 0
GoodReads duplicates: 0
Open Library duplicates: 0
GoodReads shape: (1004, 13)
Open Library shape: (1000, 15)


In [51]:
df_goodreads.to_csv("goodreads_1000_full.csv", index=False)
df_api.to_csv("openlibrary_1000.csv", index=False)
print("Both datasets saved and ready!")

Both datasets saved and ready!


In [56]:
#concatenating the two datasets together and saving 

df_combined = pd.concat([df_goodreads, df_api], ignore_index=True)
print(df_combined.shape)
print(df_combined.columns.tolist())
print(df_combined["source"].value_counts())

(2004, 18)
['rank', 'title', 'author', 'avg_rating', 'num_ratings', 'cover_img_url', 'goodreads_url', 'source', 'year_published', 'genres', 'description', 'pages', 'series', 'isbn', 'publisher', 'language', 'ol_key', 'google_url']
source
GoodReads       1004
Open Library    1000
Name: count, dtype: int64


In [57]:
df_combined.to_csv("books_combined.csv", index=False)
print("Saved to books_combined.csv!")

Saved to books_combined.csv!
